In [1]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import yfinance as yf

from saber import metaLib as mtlib

#PA = pa.PA(login = "7933713",pw = '1523@Rocket')
from saber import PyFolio as pf
from saber import utilities as ut
from saber import mailer as mail

from saber import metaData as mtd
from saber import riskEngine as risk
from saber import PcEngine as pe
from saber import implementationEngine as ie
from saber import PyFolio as pyf
from saber import PerformanceAnalytics as pa
import os
    

%load_ext autoreload
%autoreload 2

#mtw = mtlib.meta5_wrapper()

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
mtw = mtlib.meta5_wrapper(login = 7933713,pw = '1523@Rocket')
PerfA = pa.PA(mtw)
rkm = risk.riskMgr(mtw)
mtd.data(mtw).update_tick_data()
mtd.data(mtw).update_trade_sized_close()

VIX_F6  download failed
VIX_J6  download failed
VIX_K6  download failed
VIX_M6  download failed
VIX_N6  download failed
VIX_Q6  download failed
VIX_U6  download failed
VIX_V6  download failed
VIX_X6  download failed
VIX_Z6  download failed


In [102]:
def update_realized_pnl(PerfA, directory, start_date="2026-01-01"):
    import os
    
    today = pd.Timestamp.utcnow().tz_convert("Asia/Singapore")
    today_str = today.strftime("%Y-%m-%d")
    
    if start_date is None:
        start_date = today_str
    
    # Generate all dates between start_date and today
    date_range = pd.date_range(start=start_date, end=today_str, freq='D')
    
    for date in date_range:
        date_str = date.strftime("%Y-%m-%d")
        filename = f"dgm_realized_pnl_{date_str}.csv"
        filepath = os.path.join(directory, filename)
        
        # Skip if file already exists
        if os.path.exists(filepath) and date_str != today_str:
            print(f"Already exists: {filename} - skipping")
            continue
        
        try:
            # Get realized PnL for that specific date
            df = PerfA.get_realized_pnl_by_date(date.to_pydatetime())
            
            # Skip if empty or no actual PnL
            if df is None or df.empty:
                print(f"No realized trades on {date_str} - skipping")
                continue
            
            if df.drop(columns='TOTAL', errors='ignore').sum().sum() == 0:
                print(f"No realized trades on {date_str} - skipping")
                continue
            
            df.to_csv(filepath, index_label='Date')
            print(f"Saved: {filename}")
            
        except Exception as e:
            print(f"Error on {date_str}: {e} - skipping")
            continue



def update_floating_pnl(PerfA, directory):
    import os
    
    today = pd.Timestamp.utcnow().tz_convert("Asia/Singapore").strftime("%Y-%m-%d")
    filename = f"dgm_floating_pnl_{today}.csv"
    filepath = os.path.join(directory, filename)
    
    df = PerfA.get_acc_floating_pnl()
    
    if df.empty or df.drop(columns='TOTAL', errors='ignore').sum().sum() == 0:
        print(f"No open positions - skipping save")
        return
    
    df.to_csv(filepath, index_label='Date')
    print(f"Saved: {filename}")
    return df

def update_portfolio_var(rkm, directory):
    import os
    
    today = pd.Timestamp.utcnow().tz_convert("Asia/Singapore").strftime("%Y-%m-%d")
    filename = f"dgm_portfolio_var_{today}.csv"
    filepath = os.path.join(directory, filename)
    
    var = rkm.get_portfolio_var()
    
    # Wrap in dataframe with today's date if not already a dataframe
    if not isinstance(var, pd.DataFrame):
        df = pd.DataFrame({'VaR': [var]}, index=[today])
        df.index.name = 'Date'
    else:
        df = var.copy()
        df.index = [today]
        df.index.name = 'Date'
    
    if df.empty:
        print(f"No VaR data - skipping save")
        return
    
    df.to_csv(filepath, index_label='Date')
    print(f"Saved: {filename}")
    return df

In [103]:
def build_pnl_from_files(directory):
    import os
    import glob
    
    floating_dir = os.path.join(directory, 'floating_pnl')
    realized_dir = os.path.join(directory, 'realized_pnl')
    
    # Read all floating PnL files
    floating_files = sorted(glob.glob(os.path.join(floating_dir, '*.csv')))
    floating_dfs = []
    for f in floating_files:
        df = pd.read_csv(f, index_col='Date')
        df.drop(columns='TOTAL', errors='ignore', inplace=True)
        df.index = pd.to_datetime(df.index, format='mixed', dayfirst=True).strftime('%Y-%m-%d')
        floating_dfs.append(df)
    
    # Read all realized PnL files
    realized_files = sorted(glob.glob(os.path.join(realized_dir, '*.csv')))
    realized_dfs = []
    for f in realized_files:
        df = pd.read_csv(f, index_col='Date')
        df.drop(columns='TOTAL', errors='ignore', inplace=True)
        df.index = pd.to_datetime(df.index, format='mixed', dayfirst=True).strftime('%Y-%m-%d')
        realized_dfs.append(df)
    
    # Combine floating
    if len(floating_dfs) > 0:
        floating_combined = pd.concat(floating_dfs, axis=0, join='outer').fillna(0)
        floating_combined = floating_combined.groupby(floating_combined.index).sum()
    else:
        floating_combined = pd.DataFrame()
    
    # Combine realized (daily values, not cumulated yet)
    if len(realized_dfs) > 0:
        realized_combined = pd.concat(realized_dfs, axis=0, join='outer').fillna(0)
        realized_combined = realized_combined.groupby(realized_combined.index).sum()
    else:
        realized_combined = pd.DataFrame()
    
    # Get all dates from both sources
    all_dates = sorted(set(
        floating_combined.index.tolist() + realized_combined.index.tolist()
    ))
    
    # Align columns
    all_cols = list(set(
        floating_combined.columns.tolist() + realized_combined.columns.tolist()
    ))
    
    # Reindex BEFORE cumsum - fill missing realized dates with 0
    if len(realized_combined) > 0:
        realized_combined = realized_combined.reindex(index=all_dates, columns=all_cols, fill_value=0)
        realized_combined = realized_combined.sort_index()
        # Now cumsum - missing dates contribute 0 but carry forward previous cumulative
        realized_cumulative = realized_combined.cumsum()
    else:
        realized_cumulative = pd.DataFrame(0, index=all_dates, columns=all_cols)
    
    # Reindex floating - use ffill for missing dates (carry forward last known floating value)
    if len(floating_combined) > 0:
        floating_combined = floating_combined.reindex(index=all_dates, columns=all_cols)
        floating_combined = floating_combined.sort_index().ffill().fillna(0)
    else:
        floating_combined = pd.DataFrame(0, index=all_dates, columns=all_cols)
    
    # Combined PnL = cumulative realized + floating
    pnl = realized_cumulative + floating_combined
    pnl['TOTAL'] = pnl.sum(axis=1)
    
    # Drop duplicates just in case
    pnl = pnl[~pnl.index.duplicated(keep='first')]
    
    return pnl

In [104]:
update_floating_pnl(PerfA,r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\discretionary_macro\floating_pnl")
update_realized_pnl(PerfA,r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\discretionary_macro\realized_pnl")
update_portfolio_var(rkm,r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\discretionary_macro\var")

Saved: dgm_floating_pnl_2026-02-13.csv
Error on 2026-01-01: "None of [Index(['2026-01-01'], dtype='object', name='Date')] are in the [index]" - skipping
Error on 2026-01-02: "None of [Index(['2026-01-02'], dtype='object', name='Date')] are in the [index]" - skipping
Error on 2026-01-03: "None of [Index(['2026-01-03'], dtype='object', name='Date')] are in the [index]" - skipping
Error on 2026-01-04: "None of [Index(['2026-01-04'], dtype='object', name='Date')] are in the [index]" - skipping
Error on 2026-01-05: "None of [Index(['2026-01-05'], dtype='object', name='Date')] are in the [index]" - skipping
Error on 2026-01-06: "None of [Index(['2026-01-06'], dtype='object', name='Date')] are in the [index]" - skipping
Error on 2026-01-07: "None of [Index(['2026-01-07'], dtype='object', name='Date')] are in the [index]" - skipping
Error on 2026-01-08: "None of [Index(['2026-01-08'], dtype='object', name='Date')] are in the [index]" - skipping
Error on 2026-01-09: "None of [Index(['2026-01-09

,VaR
Date,
2026-02-13,-2801.225


In [105]:
pnl_df = build_pnl_from_files(r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\discretionary_macro")
pnl_df.plot()

<Axes: xlabel='Date'>

## Portfolio VaR

In [106]:
def build_var_timeseries(directory):
    """
    Read all VaR CSVs from directory and construct a timeseries
    
    Parameters:
    -----------
    directory : str
        Path to the var folder
    
    Returns:
    --------
    pd.DataFrame
        Timeseries with date as index and VaR as values
    """
    import os
    import glob
    
    var_files = sorted(glob.glob(os.path.join(directory, '*.csv')))
    
    dfs = []
    for f in var_files:
        df = pd.read_csv(f, index_col='Date')
        dfs.append(df)
    
    if len(dfs) > 0:
        var_ts = pd.concat(dfs)
        var_ts = var_ts.groupby(var_ts.index).last()  # Keep last if duplicate dates
        var_ts.index = pd.to_datetime(var_ts.index, format='mixed', dayfirst=True)
        var_ts.sort_index(inplace=True)
    else:
        var_ts = pd.DataFrame(columns=['VaR'])
    
    return var_ts

In [107]:
var_df = -build_var_timeseries(r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\discretionary_macro\var")#.plot()
var_df.plot(kind = 'bar')

<Axes: xlabel='Date'>

### Get Positions

In [108]:
def get_risk_positions_notional(mtw):
    """
    Get notional value of all open positions
    Notional = lot_size * contract_size * current_price
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with position details and notional values per asset
    """
    positions = mtw.mt5.positions_get()
    
    if positions is None or len(positions) == 0:
        print("No open positions")
        return pd.DataFrame(columns=['Volume', 'Contract Size', 'Price', 'Direction', 'Notional'])
    
    data = {}
    exclude_assets = ['USDJPY',"EURUSD","AUDUSD","GBPUSD","USDMXN","AUDJPY"]
    for p in positions:
        symbol = p.symbol
        if symbol in exclude_assets:
            continue
        volume = p.volume
        price = p.price_current
        contract_size = mtw.get_symbol_info(symbol).trade_contract_size
        direction = 1 if p.type == 0 else -1  # 0 = Buy, 1 = Sell
        notional = volume * contract_size * price * direction
        
        if symbol in data:
            data[symbol]['Volume'] += volume * direction
            data[symbol]['Notional'] += notional
        else:
            data[symbol] = {
                'Volume': volume * direction,
                'Contract Size': contract_size,
                'Price': price,
                'Direction': 'Long' if direction == 1 else 'Short',
                'Notional': notional
            }
    
    df = pd.DataFrame(data).T
    df.index.name = 'Symbol'


    
    return df

# Usage:
# notional_df = get_positions_notional()


In [109]:
get_risk_positions_notional(mtw)

,Volume,Contract Size,Price,Direction,Notional
Symbol,,,,,
VIX_G6,150.0,1.0,19.94,Long,2991.0
USTEC,-0.7,1.0,24809.9,Short,-17366.93
XOM.NYSE-24,32.0,1.0,153.82,Long,4922.24
JPM.NYSE,-16.0,1.0,302.36,Short,-4837.76
XBRUSD,10.0,100.0,68.2,Long,68200.0
BP.LSE,-780.0,1.0,4.5737,Short,-3567.486
USDCAD,0.5,100000.0,1.36187,Long,68093.5


In [110]:
def get_non_currency_longs(mtw):
    pos_df = get_risk_positions_notional(mtw)
    
    # FX pairs are 6 characters with no "." (e.g., EURUSD, USDJPY, AUDUSD)
    fx_pairs  = ['USDJPY','AUDJPY',"EURUSD","EURAUD","AUDNZD","GBPUSD","USDMXN","USDZAR","USDCAD"]
    fx_symbols = [sym for sym in pos_df.index if sym in fx_pairs]
    
    # Filter: long positions that are NOT FX
    non_fx_longs = pos_df[(pos_df['Notional'] > 0) & (~pos_df.index.isin(fx_symbols))]
    
    return non_fx_longs

    
def get_non_currency_shorts(mtw):
    pos_df = get_risk_positions_notional(mtw)
    
    # FX pairs are 6 characters with no "." (e.g., EURUSD, USDJPY, AUDUSD)
    fx_pairs  = ['USDJPY','AUDJPY',"EURUSD","EURAUD","AUDNZD","GBPUSD","USDMXN","USDZAR","USDCAD"]
    fx_symbols = [sym for sym in pos_df.index if sym in fx_pairs]
    
    # Filter: long positions that are NOT FX
    non_fx_shorts = pos_df[(pos_df['Notional'] < 0) & (~pos_df.index.isin(fx_symbols))]
    
    return non_fx_shorts

def plot_non_fx_pie_charts(mtw):
    non_fx_longs = get_non_currency_longs(mtw)
    non_fx_shorts = get_non_currency_shorts(mtw)
    
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    
    if len(non_fx_longs) > 0:
        longs_pct = non_fx_longs['Notional'] / non_fx_longs['Notional'].sum() * 100
        axes[0].pie(longs_pct, labels=longs_pct.index, autopct='%1.1f%%', startangle=90)
        axes[0].set_title('Non-FX Longs')
    else:
        axes[0].text(0.5, 0.5, 'No Long Positions', ha='center', va='center')
        axes[0].set_title('Non-FX Longs')
    
    if len(non_fx_shorts) > 0:
        shorts_pct = non_fx_shorts['Notional'].abs() / non_fx_shorts['Notional'].abs().sum() * 100
        axes[1].pie(shorts_pct, labels=shorts_pct.index, autopct='%1.1f%%', startangle=90)
        axes[1].set_title('Non-FX Shorts')
    else:
        axes[1].text(0.5, 0.5, 'No Short Positions', ha='center', va='center')
        axes[1].set_title('Non-FX Shorts')
    
    plt.tight_layout()
    plt.show()

# Usage:
# plot_non_fx_pie_charts(mtw)
def get_expanding_sharpe(pnl_df, column='TOTAL', annualization_factor=252):
    """
    Calculate expanding (cumulative) Sharpe ratio over time
    
    Parameters:
    -----------
    pnl_df : pd.DataFrame
        Cumulative PnL dataframe
    column : str
        Column to calculate Sharpe on (default 'TOTAL')
    annualization_factor : int
        Trading days per year (default 252)
    
    Returns:
    --------
    pd.Series
        Expanding Sharpe ratio at each date
    """
    # Get daily returns from cumulative PnL
    daily_returns = pnl_df[column].diff()
    
    # Expanding mean and std
    expanding_mean = daily_returns.expanding(min_periods=2).mean()
    expanding_std = daily_returns.expanding(min_periods=2).std()
    
    # Annualized Sharpe ratio
    sharpe = (expanding_mean / expanding_std) * np.sqrt(annualization_factor)
    sharpe.name = 'Sharpe Ratio'
    
    return sharpe

In [111]:
plot_non_fx_pie_charts(mtw)

C:\Users\kmavy\AppData\Local\Temp\ipykernel_21268\2587573639.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [112]:
rkm.get_portfolio_var()

-2801.224999999996

In [113]:
rkm.get_indiv_pos_var()

{'BP.LSE': -110.328,
 'JPM.NYSE': -78.90564102564102,
 'USDCAD': -572.7211538461543,
 'USTEC': -248.78430769230764,
 'VIX_G6': -78.03703703703702,
 'XBRUSD': -3190.448717948718,
 'XOM.NYSE-24': -37.45684210526317}

In [114]:
def send_dgm_summary_email(pnl_df, var_ts, mtw, recipient):
    """
    Send discretionary macro portfolio summary email with charts
    
    Parameters:
    -----------
    pnl_df : pd.DataFrame
        Cumulative PnL dataframe (from build_pnl_from_files)
    var_ts : pd.DataFrame
        VaR timeseries (from build_var_timeseries)
    mtw : meta5_wrapper
        MT5 wrapper for getting positions
    recipient : str
        Email address to send to
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import base64
    import tempfile
    import os
    from email.mime.multipart import MIMEMultipart
    from email.mime.text import MIMEText
    from email.mime.image import MIMEImage
    from saber import mailer
    
    today = pd.Timestamp.utcnow().tz_convert("Asia/Singapore").strftime("%Y-%m-%d")
    chart_paths = []
    temp_dir = tempfile.mkdtemp()
    
    # Sort PnL by date
    df = pnl_df.copy()
    df.index = pd.to_datetime(df.index, format='mixed', dayfirst=True)
    df.sort_index(inplace=True)
    
    # --- Chart 1: Cumulative PnL ---
    fig, ax = plt.subplots(figsize=(8, 3))
    df['TOTAL'].plot(ax=ax)
    ax.set_title('Cumulative PnL')
    ax.set_ylabel('PnL (USD)')
    ax.grid(True, alpha=0.3)
    path1 = os.path.join(temp_dir, 'cum_pnl.png')
    fig.savefig(path1, bbox_inches='tight', dpi=100)
    plt.close(fig)
    chart_paths.append(path1)
    
    # --- Chart 2: Daily VaR ---
    fig, ax = plt.subplots(figsize=(8, 3))
    var_ts.plot(ax=ax, kind = 'bar',color = 'red')
    ax.set_title('Daily Portfolio VaR (95% CVaR)')
    ax.set_ylabel('VaR (USD)')
    ax.grid(True, alpha=0.3)
    path2 = os.path.join(temp_dir, 'var.png')
    fig.savefig(path2, bbox_inches='tight', dpi=100)
    plt.close(fig)
    chart_paths.append(path2)
    
    # --- Chart 3: Non-FX Long/Short Exposure ---
    non_fx_longs = get_non_currency_longs(mtw)
    non_fx_shorts = get_non_currency_shorts(mtw)
    
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    
    if len(non_fx_longs) > 0:
        longs_pct = non_fx_longs['Notional'] / non_fx_longs['Notional'].sum() * 100
        axes[0].pie(longs_pct, labels=longs_pct.index, autopct='%1.1f%%', startangle=90)
        axes[0].set_title('Non-FX Longs')
    else:
        axes[0].text(0.5, 0.5, 'No Long Positions', ha='center', va='center')
        axes[0].set_title('Non-FX Longs')
    
    if len(non_fx_shorts) > 0:
        shorts_pct = non_fx_shorts['Notional'].abs() / non_fx_shorts['Notional'].abs().sum() * 100
        axes[1].pie(shorts_pct, labels=shorts_pct.index, autopct='%1.1f%%', startangle=90)
        axes[1].set_title('Non-FX Shorts')
    else:
        axes[1].text(0.5, 0.5, 'No Short Positions', ha='center', va='center')
        axes[1].set_title('Non-FX Shorts')
    
    plt.tight_layout()
    path3 = os.path.join(temp_dir, 'exposure.png')
    fig.savefig(path3, bbox_inches='tight', dpi=100)
    plt.close(fig)
    chart_paths.append(path3)
    
    # --- Chart 4: Expanding Sharpe Ratio ---
    expanding_sharpe = get_expanding_sharpe(df)
    
    fig, ax = plt.subplots(figsize=(8, 3))
    expanding_sharpe.plot(ax=ax)
    ax.set_title('Expanding Sharpe Ratio (Annualized)')
    ax.set_ylabel('Sharpe Ratio')
    ax.grid(True, alpha=0.3)
    path4 = os.path.join(temp_dir, 'sharpe.png')
    fig.savefig(path4, bbox_inches='tight', dpi=100)
    plt.close(fig)
    chart_paths.append(path4)
    
    # --- Key Metrics ---
    total_pnl = df['TOTAL'].iloc[-1]
    latest_var = var_ts['VaR'].iloc[-1] if len(var_ts) > 0 else 0
    latest_sharpe = expanding_sharpe.iloc[-1] if len(expanding_sharpe.dropna()) > 0 else 0
    long_notional = non_fx_longs['Notional'].sum() if len(non_fx_longs) > 0 else 0
    short_notional = non_fx_shorts['Notional'].sum() if len(non_fx_shorts) > 0 else 0
    
    # --- Build Email ---
    msg = MIMEMultipart('related')
    msg['Subject'] = f'Discretionary Macro Summary - {today}'
    msg['To'] = recipient
    
    html = f"""
    <html>
    <body style="font-family: Arial, sans-serif;">
        <h2>Discretionary Macro Summary - {today}</h2>
        
        <h3>Key Metrics</h3>
        <table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse;">
            <tr><td><b>Total PnL</b></td><td>{total_pnl:,.2f}</td></tr>
            <tr><td><b>Portfolio VaR (95% CVaR)</b></td><td>{latest_var:,.2f}</td></tr>
            <tr><td><b>Sharpe Ratio</b></td><td>{latest_sharpe:,.2f}</td></tr>
            <tr><td><b>Non-FX Long Notional</b></td><td>{long_notional:,.2f}</td></tr>
            <tr><td><b>Non-FX Short Notional</b></td><td>{short_notional:,.2f}</td></tr>
            <tr><td><b>Non-FX Net Notional</b></td><td>{long_notional + short_notional:,.2f}</td></tr>
        </table>
        
        <h3>Cumulative PnL</h3>
        <img src="cid:chart1" width="600">
        
        <h3>Daily Portfolio VaR</h3>
        <img src="cid:chart2" width="600">
        
        <h3>Non-FX Exposure Breakdown</h3>
        <img src="cid:chart3" width="600">
        
        <h3>Expanding Sharpe Ratio</h3>
        <img src="cid:chart4" width="600">
    </body>
    </html>
    """
    
    msg.attach(MIMEText(html, 'html'))
    
    for i, path in enumerate(chart_paths, 1):
        with open(path, 'rb') as f:
            img = MIMEImage(f.read())
            img.add_header('Content-ID', f'<chart{i}>')
            msg.attach(img)
    
    # Send via Gmail API
    service = mailer.get_service()
    raw_msg = base64.urlsafe_b64encode(msg.as_bytes()).decode()
    service.users().messages().send(userId='me', body={'raw': raw_msg}).execute()
    
    # Cleanup
    for path in chart_paths:
        os.remove(path)
    
    print(f"Discretionary Macro summary sent to {recipient}")

# Usage:
pnl_df = build_pnl_from_files(r'C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\discretionary_macro')
var_ts = build_var_timeseries(r'C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\discretionary_macro\var')
send_dgm_summary_email(pnl_df, var_ts, mtw, 'kmavyrle@gmail.com')


Discretionary Macro summary sent to kmavyrle@gmail.com


In [115]:
pnl_df

,EURAUD,EURUSD,USDCAD,AUDJPY,VIX_G6,BP.LSE,XBRUSD,UST30Y_H6,XAUUSD,AUDNZD,XOM.NYSE-24,USTEC,JPM.NYSE,XAGUSD,USDJPY,GBPUSD,BTCUSD,TOTAL
Date,,,,,,,,,,,,,,,,,,
2026-02-01,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00
2026-02-02,0.00,0.0,0.00,0.00,-102.0,0.0,-344.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,-446.00
2026-02-03,697.11,0.0,0.00,4.37,-147.0,0.0,-6.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,66.18,0.0,0.00,614.66
2026-02-04,744.90,0.0,0.00,46.49,-27.0,0.0,730.84,0.0,0.00,-131.29,0.00,80.89,0.00,0.00,241.83,0.0,0.00,1686.66
2026-02-05,462.56,0.0,0.00,-136.40,280.5,0.0,898.84,0.0,-13.42,-161.32,0.00,142.48,0.00,-103.80,162.83,0.0,0.00,1532.27
2026-02-06,462.56,0.0,0.00,-136.40,343.5,0.0,1102.84,0.0,-13.42,338.68,0.00,-38.50,0.00,-103.80,162.83,-45.5,0.00,2072.79
2026-02-07,462.56,0.0,0.00,-136.40,-21.0,0.0,1016.84,12.0,-13.42,338.68,0.00,-373.82,0.00,-103.80,306.37,-146.5,0.00,1341.51
2026-02-09,462.56,0.0,0.00,-136.40,-21.0,0.0,1016.84,-20.0,-13.42,338.68,0.00,-373.82,0.00,-103.80,99.61,-146.5,0.00,1102.75
2026-02-10,462.56,0.0,0.00,-136.40,-21.0,0.0,1016.84,-20.0,-13.42,338.68,0.00,-373.82,0.00,-103.80,99.61,-423.2,0.00,826.05


In [ ]:
# Usage:
mtw = mtlib.meta5_wrapper()
PerfA = pa.PA(mtw)
rkm = risk.riskMgr(mtw)
update_floating_pnl(PerfA,r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\systematic\floating_pnl")
update_realized_pnl(PerfA,r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\systematic\realized_pnl")
update_portfolio_var(rkm,r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\systematic\var")
#pnl_df = build_pnl_from_files(r'C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\systematic')
pnl_df = pd.DataFrame()
for file in os.listdir(r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\systematic\realized_pnl"):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(r"C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\systematic\realized_pnl", file), index_col='Date')
        # Normalize index to consistent string format
        df.index = pd.to_datetime(df.index, format='mixed', dayfirst=True).strftime('%Y-%m-%d')
        pnl_df = pnl_df.add(df, fill_value=0)


pnl_df.index = pd.to_datetime(pnl_df.index,format = 'mixed',dayfirst= True)
#pnl_df = pnl_df[~pnl_df.index.duplicated(keep='first')]
pnl_df = pnl_df.sort_index().fillna(0).cumsum()
pnl_df
var_ts = build_var_timeseries(r'C:\Users\kmavy\Documents\mydocs\My Docs\Investments\investments-app\systematic\var')
send_dgm_summary_email(pnl_df, var_ts, mtw, 'kmavyrle@gmail.com')

Saved: dgm_floating_pnl_2026-02-13.csv
Already exists: dgm_realized_pnl_2026-01-01.csv - skipping
Already exists: dgm_realized_pnl_2026-01-02.csv - skipping
Already exists: dgm_realized_pnl_2026-01-03.csv - skipping
Error on 2026-01-04: "None of [Index(['2026-01-04'], dtype='object', name='Date')] are in the [index]" - skipping
No realized trades on 2026-01-05 - skipping
Already exists: dgm_realized_pnl_2026-01-06.csv - skipping
Already exists: dgm_realized_pnl_2026-01-07.csv - skipping
Already exists: dgm_realized_pnl_2026-01-08.csv - skipping
Error on 2026-01-09: "None of [Index(['2026-01-09'], dtype='object', name='Date')] are in the [index]" - skipping
Already exists: dgm_realized_pnl_2026-01-10.csv - skipping
Error on 2026-01-11: "None of [Index(['2026-01-11'], dtype='object', name='Date')] are in the [index]" - skipping
Already exists: dgm_realized_pnl_2026-01-12.csv - skipping
Error on 2026-01-13: "None of [Index(['2026-01-13'], dtype='object', name='Date')] are in the [index]" 